In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import numpy as np

In [2]:
env_name = "Hopper-v5"
# gamma =??
lr = 0.001
# clip_eps =??
# epochs = ??
# steps_per_epoch =?? 
# batch_size = ??
# entropy_coef = ??
hidden_dim = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
env = gym.make(env_name)
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])

In [4]:
obs = env.reset()[0]

In [5]:
obs_dim, act_dim, max_action

(11, 3, 1.0)

In [6]:
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim, device=device),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim, device=device)
        )
        self.log_std = nn.Parameter(torch.zeros(act_dim))  

    def forward(self, x):
        mu = self.net(x)
        std = torch.exp(self.log_std)
        return mu, std

# --- Réseau Value ---
class Value(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim, device=device),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1, device=device)
        )

    def forward(self, x):
        return self.net(x)

# --- Initialisation ---
policy = Policy().to(device)
value = Value().to(device)
optimizer_policy = optim.Adam(policy.parameters(), lr=lr)
optimizer_value = optim.Adam(value.parameters(), lr=lr)

c:\Users\natha\Documents\Git\5th-year-polytech\Reinforcement Learning\env\Lib\site-packages\torch\nn\modules\linear.py:109: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  torch.empty((out_features, in_features), **factory_kwargs)


In [11]:
obs_tensor = torch.FloatTensor(obs).to(device)
mu, std = policy(obs_tensor)
dist = torch.distributions.Normal(mu, std)
act = dist.sample()
logp = dist.log_prob(act)
val = value(obs_tensor)
# Scale action to environment
act_clamped = nn.Tanh()(act)
next_obs, rew, terminated, truncated, _ = env.step(act_clamped.cpu().detach().numpy())
done = terminated or truncated

In [12]:
next_obs, rew, terminated, truncated

(array([ 1.24620889e+00,  5.58480511e-04,  1.44000480e-03,  9.99207754e-04,
        -8.70664317e-03, -2.85260338e-02, -1.48543979e-01, -1.08584463e-01,
         1.51794858e-02, -2.62876538e-01, -5.07401772e-02]),
 np.float64(0.9864448269134165),
 False,
 False)